In [242]:
import sys
import statistics
from pathlib import Path
from collections import defaultdict, Counter

In [240]:
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [4]:
import pandas as pd
import altair as alt

In [5]:
from rich import print
from rich.progress import track

In [6]:
nlp = spacy.load('en_core_web_sm')

In [7]:
this_dir = Path("__file__").parent.absolute()
sys.path.append(this_dir.parent)
sys.path.append(str(this_dir.parent / "newshomepages"))

In [8]:
extracts_dir = this_dir.parent / "extracts" / "csv"

Read in data

In [9]:
drudge_df = pd.read_csv(extracts_dir / "drudge-hyperlinks-analysis.csv", parse_dates=["earliest_date"])

In [10]:
drudge_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5988 entries, 0 to 5987
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   text           5988 non-null   object        
 1   url            5988 non-null   object        
 2   earliest_date  5988 non-null   datetime64[ns]
 3   is_story       5988 non-null   bool          
 4   domain         5988 non-null   object        
dtypes: bool(1), datetime64[ns](1), object(3)
memory usage: 193.1+ KB


In [11]:
drudge_df.earliest_date.describe()

/tmp/ipykernel_997455/4166057540.py:1: FutureWarning: Treating datetime data as categorical rather than numeric in `.describe` is deprecated and will be removed in a future version of pandas. Specify `datetime_is_numeric=True` to silence this warning and adopt the future behavior now.
  drudge_df.earliest_date.describe()


count                    5988
unique                     90
top       2022-08-06 00:00:00
freq                      274
first     2022-08-06 00:00:00
last      2022-11-03 00:00:00
Name: earliest_date, dtype: object

In [12]:
drudge_df.is_story.value_counts()

True     5742
False     246
Name: is_story, dtype: int64

In [13]:
drudge_df.is_story.value_counts(normalize=True)

True     0.958918
False    0.041082
Name: is_story, dtype: float64

Filer down to stories

In [14]:
story_df = drudge_df[drudge_df.is_story].copy()

In [15]:
story_df.to_csv("./drudge-stories.csv", index=False)

## Preprocess

Cut `...`

In [16]:
story_df.text = story_df.text.str.replace(r"\.{2,}", "", regex=True)

Extract documents

In [17]:
documents = sorted(list(story_df.text.unique()))

Extract named entities

In [18]:
def get_named_entities(text: str) -> dict:
    doc = nlp(text)
    return [dict(
        text=text,
        type=e.label_,
        name=e.text,
    ) for e in doc.ents]

In [19]:
entity_list = []
for doc in track(documents):
    entity_list.extend(get_named_entities(doc))

Output()

In [20]:
entity_df = pd.DataFrame(entity_list)

In [21]:
entity_df.head()

,text,type,name
0,#MeToo 5 Years Later: No One's Fully Returned ...,ORG,MeToo 5 Years Later: No One's
1,$$49.3 MILLION VERDICT,MONEY,$49.3 MILLION
2,$$49.3 MILLION VERDICT,ORG,VERDICT
3,$1.6 Billion Defamation Suit Tests FOX CEO's V...,MONEY,$1.6 Billion
4,"$15 French Fries, $18 Sandwiches: Inflation Hi...",MONEY,15


Exclude ordinal entities

In [28]:
qualified_df = entity_df[~entity_df.type.isin(["ORDINAL"])]

In [32]:
entity_counts = qualified_df.groupby(["name", "type"]).size().rename("n").reset_index()

In [33]:
entity_counts['percent'] = entity_counts.n / len(documents)

In [35]:
entity_counts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3280 entries, 0 to 3279
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   name     3280 non-null   object 
 1   type     3280 non-null   object 
 2   n        3280 non-null   int64  
 3   percent  3280 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 102.6+ KB


In [34]:
entity_counts.sort_values("n", ascending=False).head(25)

,name,type,n,percent
691,Biden,PERSON,61,0.010751
901,China,GPE,57,0.010046
2439,Russia,GPE,55,0.009693
2941,USA,GPE,54,0.009517
2393,Republicans,NORP,52,0.009165
2315,Putin,PERSON,50,0.008812
2862,Trump,ORG,45,0.007931
2948,Ukraine,GPE,44,0.007755
1562,Iran,GPE,43,0.007578
1300,Florida,GPE,36,0.006345


In [36]:
entity_counts.to_csv("drudge-entity-counts.csv", index=False)

In [38]:
spacy.explain("GPE")

'Countries, cities, states'

In [41]:
entity_df[entity_df.text.str.contains("BIDEN")]

,text,type,name
728,BIDEN BIG SUMMER,ORG,BIDEN BIG SUMMER
729,BIDEN COVID NEGATIVE; STAYS ISOLATED,ORG,BIDEN COVID NEGATIVE
730,BIDEN FREE,ORG,BIDEN FREE
731,BIDEN TAKES SWING AT CNN,ORG,CNN
732,BIDEN: TRUMP THREAT TO AMERICA,ORG,TRUMP THREAT TO AMERICA
1740,ELTON JOHN BREAKS DOWN IN TEARS AS BIDEN SURPR...,PERSON,ELTON JOHN BREAKS
1971,FOX POLL: BIDEN APPROVAL 46%,ORG,FOX
1972,FOX POLL: BIDEN APPROVAL 46%,PERCENT,46%
4820,SURPRISE BIDEN ON HILL URGING VOTES AGAINST PO...,ORG,ON HILL URGING


In [189]:
def get_lemma(text):
    doc = nlp(text.upper())
    return [dict(
        headline=text,
        word=token.text.upper(),
        lemma=token.lemma_.upper(),
        pos=token.pos_,
        is_digit=token.is_digit,
    )for token in doc if not token.is_stop and not token.is_punct]

In [190]:
lemma_list = []
for doc in track(documents):
    lemma_list.extend(get_lemma(doc))

Output()

In [197]:
lemma_df = pd.DataFrame(lemma_list)

In [198]:
lemma_df.head()

,headline,word,lemma,pos,is_digit
0,#MeToo 5 Years Later: No One's Fully Returned ...,METOO,METOO,PROPN,False
1,#MeToo 5 Years Later: No One's Fully Returned ...,5,5,NUM,True
2,#MeToo 5 Years Later: No One's Fully Returned ...,YEARS,YEAR,NOUN,False
3,#MeToo 5 Years Later: No One's Fully Returned ...,LATER,LATER,ADV,False
4,#MeToo 5 Years Later: No One's Fully Returned ...,FULLY,FULLY,ADJ,False


In [229]:
extra_stops = [
    "NEW",
    "MAN",
    "WOMAN",
    "YEAR",
    "DAY",
    "MILLION",
    "HIGH",
    "BIG",
    "RECORD",
    "HOME",
    "WORLD",
    "STATE",
    "TIME",
    "CASE",
    "LIFE",
    "AMERICAN",
]

In [230]:
qualified_df = lemma_df[
    (~lemma_df.is_digit) &
    (~lemma_df.pos.isin(["SYM", "VERB"])) &
    (~lemma_df.lemma.isin(extra_stops))
]

In [231]:
top_words = qualified_df.groupby(["lemma"]).size().rename("n").reset_index().sort_values("n", ascending=False).head(25)

In [232]:
top_words.head(25)

,lemma,n
7208,TRUMP,146
5667,REPUBLICAN,103
2272,ELECTION,95
739,BIDEN,88
7549,WAR,88
5396,PUTIN,79
7386,USA,68
112,ABORTION,68
1631,COVID,68
2118,DON,66


In [237]:
def get_top_verb(lemma):
    headline_list = list(lemma_df[lemma_df.lemma == lemma].headline.unique())
    verb_list = []
    stop_verbs = ["SAYS",]
    if lemma.upper() == "COVID":
        stop_verbs += ["TESTS"]
    if lemma.upper() == "MUSK":
        stop_verbs += ["SOCIAL"]
    for headline in headline_list:
        doc = nlp(headline)
        verb_list.extend([t.text.upper() for t in doc if t.pos_ == "VERB" if not t.is_stop and not t.is_punct and t.text.upper() not in stop_verbs])
    return Counter(verb_list).most_common(1)[0][0]

In [238]:
top_words['top_verb'] = top_words.lemma.apply(get_top_verb)

In [239]:
top_words.head(25)

,lemma,n,top_verb
7208,TRUMP,146,FACES
5667,REPUBLICAN,103,THREATEN
2272,ELECTION,95,GROWS
739,BIDEN,88,WANT
7549,WAR,88,WARNS
5396,PUTIN,79,BLOWN
7386,USA,68,WARNS
112,ABORTION,68,CLAIMS
1631,COVID,68,WARNS
2118,DON,66,STAND


In [236]:
# list(lemma_df[lemma_df.lemma == 'LIFE'].headline.unique())

In [241]:
vaderAnalyzer = SentimentIntensityAnalyzer()

In [268]:
def get_sentiment(lemma):
    headline_list = list(lemma_df[lemma_df.lemma == lemma].headline.unique())
    text = " ".join(headline_list)
    return vaderAnalyzer.polarity_scores(text)["compound"]

In [269]:
top_words['sentiment'] = top_words.lemma.apply(get_sentiment)

In [270]:
top_words.head(25)

,lemma,n,top_verb,sentiment
7208,TRUMP,146,FACES,-0.9980
5667,REPUBLICAN,103,THREATEN,-0.9966
2272,ELECTION,95,GROWS,-0.9973
739,BIDEN,88,WANT,-0.9943
7549,WAR,88,WARNS,-0.9999
5396,PUTIN,79,BLOWN,-0.9984
7386,USA,68,WARNS,-0.9958
112,ABORTION,68,CLAIMS,-0.9933
1631,COVID,68,WARNS,-0.9984
2118,DON,66,STAND,-0.9990
